In [2]:
%pip install statsmodels
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from collections import Counter
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Source Han Sans CN']  # 优先使用系统已有的中文字体

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# 1. 导入数据
# 读取清理后的数据
print("=" * 50)
print("导入数据")
print("=" * 50)

df = pd.read_csv('rent_price_final_train_dataset3.csv')

print(f"数据形状: {df.shape}")
print(f"变量数量: {len(df.columns)}")
print(f"记录数量: {len(df)}")

# 检查数据基本信息
print("\n数据基本信息:")
print(df.info())

# 显示前几行数据，确认数据正确加载
print("\n前5行数据:")
print(df.head())

# 检查列名，确认所有需要的变量都存在
print("\n数据集列名:")
print(df.columns.tolist())

导入数据
数据形状: (98899, 81)
变量数量: 81
记录数量: 98899

数据基本信息:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 98899 entries, 0 to 98898
Data columns (total 81 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Price                   98899 non-null  float64
 1   lnPrice                 98899 non-null  float64
 2   decoration              98899 non-null  int64  
 3   high_dummy              98899 non-null  float64
 4   middle_dummy            98899 non-null  float64
 5   low_dummy               98899 non-null  float64
 6   basement_dummy          98899 non-null  float64
 7   total_floor             98899 non-null  float64
 8   area                    98899 non-null  float64
 9   room_count              98899 non-null  float64
 10  hall_count              98899 non-null  float64
 11  south_dummy             98899 non-null  float64
 12  north_south_dummy       98899 non-null  float64
 13  pay_annual              98899 non-null

In [4]:
# 特征工程 - 创建衍生变量
print("\n执行特征工程...")

# 1. 创建幂次项
print("创建连续变量的幂次项...")
continuous_vars_for_power = [
    'area', 'building_age', 'greening_rate', 'plot_ratio', 'household_total', 'building_total',
    'property_fee_avg', 'room_count', 'total_floor', 'gas_fee_avg'
]

power_terms = []
for var in continuous_vars_for_power:
    if var in df.columns:
        # 平方项
        square_term = f'{var}^2'
        if square_term not in df.columns:
            df[square_term] = df[var] ** 2
            print(f"已创建平方项: {square_term}")
        else:
            print(f"平方项已存在: {square_term}")
        power_terms.append(square_term)

print(f"总共处理了 {len(power_terms)} 个幂次项")

# 2. 检查并创建城市哑变量（如果尚未创建）
if '城市' in df.columns:
    # 检查是否已存在城市哑变量
    city_dummy_cols = [col for col in df.columns if col.startswith('city_')]
    if not city_dummy_cols:
        # 创建城市哑变量
        for city_num in range(12):  # 创建0-11的城市哑变量
            col_name = f'city_{city_num}'
            df[col_name] = (df['城市'] == city_num).astype(int)
        print(f"已创建城市哑变量")
    else:
        print(f"城市哑变量已存在")
else:
    print("训练集中没有'城市'列，无法创建城市哑变量")

# 3. 检查并创建城市距离变量（如果尚未创建）
if '城市' in df.columns and '距市中心距离_km' in df.columns:
    # 检查是否已存在城市距离变量
    city_dist_cols = [col for col in df.columns if col.startswith('city_dist_')]
    if not city_dist_cols:
        # 创建城市距离变量
        for city_num in range(12):  # 创建0-11的城市距离变量
            col_name = f'city_dist_{city_num}'
            df[col_name] = 0
            # 只有当城市编号匹配时才使用实际距离
            df.loc[df['城市'] == city_num, col_name] = df.loc[df['城市'] == city_num, '距市中心距离_km']
        print(f"已创建城市距离变量")
    else:
        print(f"城市距离变量已存在")
else:
    print("训练集中缺少'城市'或'距市中心距离_km'列，无法创建城市距离变量")

# 4. 创建城市和面积的交互项
print("创建城市和面积的交互项...")

# 确保城市哑变量和面积变量存在
city_columns = [f'city_{i}' for i in range(12) if f'city_{i}' in df.columns]
area_column = 'area'

if city_columns and area_column in df.columns:
    # 删除可能已存在的交互项
    existing_interaction_cols = [col for col in df.columns if col.startswith('city_area_interaction_')]
    if existing_interaction_cols:
        df.drop(existing_interaction_cols, axis=1, inplace=True)
        print(f"已删除 {len(existing_interaction_cols)} 个已存在的交互项")
    
    # 创建城市和面积的交互项
    interaction_terms_created = 0
    for city_col in city_columns:
        interaction_col = f'city_area_interaction_{city_col}'
        df[interaction_col] = df[city_col] * df[area_column]
        interaction_terms_created += 1
        print(f"已创建交互项: {interaction_col}")
    
    print(f"总共创建了 {interaction_terms_created} 个城市-面积交互项")
    
    # 将交互项添加到后续的base_vars列表中
    interaction_terms = [f'city_area_interaction_{city_col}' for city_col in city_columns]
else:
    interaction_terms = []
    if not city_columns:
        print("警告：未找到城市哑变量，无法创建交互项")
    if area_column not in df.columns:
        print("警告：未找到面积变量，无法创建交互项")

# 5. 创建城市和面积平方的交互项（可选，增强模型灵活性）
print("创建城市和面积平方的交互项...")
area_sq_column = 'area^2'

if city_columns and area_sq_column in df.columns:
    # 删除可能已存在的交互项
    existing_interaction_cols = [col for col in df.columns if col.startswith('city_area_sq_interaction_')]
    if existing_interaction_cols:
        df.drop(existing_interaction_cols, axis=1, inplace=True)
    
    # 创建城市和面积平方的交互项
    interaction_terms_created_sq = 0
    for city_col in city_columns:
        interaction_col = f'city_area_sq_interaction_{city_col}'
        df[interaction_col] = df[city_col] * df[area_sq_column]
        interaction_terms_created_sq += 1
        print(f"已创建交互项: {interaction_col}")
    
    print(f"总共创建了 {interaction_terms_created_sq} 个城市-面积平方交互项")
    
    # 将交互项添加到后续的base_vars列表中
    interaction_terms_sq = [f'city_area_sq_interaction_{city_col}' for city_col in city_columns]
else:
    interaction_terms_sq = []
    if not city_columns:
        print("警告：未找到城市哑变量，无法创建面积平方交互项")
    if area_sq_column not in df.columns:
        print("警告：未找到面积平方变量，无法创建交互项")

# 4. 基于准确的租房变量列表定义基准模型变量
base_vars = [
    # 装修类型（注意：租房数据中是decoration，不是decoration_精装等）
    'decoration',
    # 楼层位置
    'high_dummy', 'middle_dummy', 'low_dummy', 'basement_dummy',
    # 房屋基本信息
    'total_floor', 'area', 'room_count', 'hall_count', 
    # 朝向
    'south_dummy', 'north_south_dummy',
    # 付款方式
    'pay_annual', 'pay_bi_monthly', 'pay_monthly', 'pay_quarterly', 'pay_semi_annual',
    # 租赁类型
    'rent_type_shared',
    # 电梯
    'elevator_yes',
    # 水电燃气类型
    'water_civil', 'water_commercial', 'electricity_civil', 'electricity_commercial', 'gas_yes',
    # 供暖类型
    'heating_self',
    # 租期信息
    'lease_avg_months',
    # 配套设施
    'facility_洗衣机', 'facility_空调', 'facility_衣柜', 'facility_电视', 'facility_冰箱',
    'facility_热水器', 'facility_床', 'facility_天然气', 'facility_暖气', 'facility_宽带',
    # 建筑信息
    'building_age', 'household_total', 'building_total',
    'greening_rate', 'plot_ratio', 'property_fee_avg',
    # 结构类型
    'structure_tower', 'structure_slab', 'structure_combined', 'structure_bungalow',
    # 费用信息
    'gas_fee_avg', 'heating_fee_avg',
    # 停车位
    'parking_spots',
    # 其他
    'property_phone_yes',
    # 城市信息
    '城市', '距市中心距离_km',
    # 城市哑变量
    'city_0', 'city_1', 'city_2', 'city_3', 'city_4', 'city_5',
    'city_6', 'city_7', 'city_8', 'city_9', 'city_10', 'city_11',
    # 城市距离变量
    'city_dist_0', 'city_dist_1', 'city_dist_2', 'city_dist_3', 'city_dist_4', 'city_dist_5',
    'city_dist_6', 'city_dist_7', 'city_dist_8', 'city_dist_9', 'city_dist_10', 'city_dist_11',
    # 面积平方项
    'area^2'
]

# 添加交互项到base_vars
for term in interaction_terms:
    if term not in base_vars:
        base_vars.append(term)
        print(f"已添加交互项到变量列表: {term}")

for term in interaction_terms_sq:
    if term not in base_vars:
        base_vars.append(term)
        print(f"已添加交互项到变量列表: {term}")

# 确保幂次项在base_vars中
for term in power_terms:
    if term not in base_vars:
        base_vars.append(term)
        print(f"已添加幂次项到变量列表: {term}")

print("\n检查变量存在性...")
# 检查哪些变量实际存在
existing_base_vars = [col for col in base_vars if col in df.columns]
print(f"基准模型变量数量: {len(existing_base_vars)}/{len(base_vars)}")

# 列出缺失的变量
missing_vars = set(base_vars) - set(existing_base_vars)
if missing_vars:
    print(f"缺失的变量: {missing_vars}")

# 检查目标变量是否存在
if 'lnPrice' not in df.columns:
    if 'Price' in df.columns:
        df['lnPrice'] = np.log(df['Price'])
        print("已创建lnPrice变量")
    else:
        print("错误：数据集中没有Price或lnPrice变量")
        exit()

# 准备数据 - 使用existing_base_vars
X = df[existing_base_vars]
y = df['lnPrice']

# 处理缺失值
print(f"\n处理缺失值...")
X_filled = X.copy()
y_filled = y.copy()

# 填充自变量的缺失值
for col in X_filled.columns:
    if X_filled[col].isnull().sum() > 0:
        null_count = X_filled[col].isnull().sum()
        if X_filled[col].dtype in ['float64', 'int64']:
            median_val = X_filled[col].median()
            X_filled[col] = X_filled[col].fillna(median_val)
            print(f"  - {col}: {null_count} 个缺失值已用中位数填充")
        else:
            mode_val = X_filled[col].mode()[0] if not X_filled[col].mode().empty else 0
            X_filled[col] = X_filled[col].fillna(mode_val)
            print(f"  - {col}: {null_count} 个缺失值已用众数填充")

# 填充因变量的缺失值（如果有）
if y_filled.isnull().sum() > 0:
    null_count = y_filled.isnull().sum()
    median_y = y_filled.median()
    y_filled = y_filled.fillna(median_y)
    print(f"  - lnPrice: {null_count} 个缺失值已用中位数填充")

print(f"处理缺失值后数据形状: {X_filled.shape}")

# 划分训练集和测试集
print("\n划分训练集和测试集...")
X_train, X_test, y_train, y_test = train_test_split(
    X_filled, y_filled, test_size=0.2, random_state=42
)
print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")

# 添加常数项
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

# 建立OLS模型
print("\n正在拟合OLS模型...")
ols_model = sm.OLS(y_train, X_train_const)
ols_results = ols_model.fit()

# 样本内预测
y_train_pred_ln = ols_results.predict(X_train_const)
y_train_pred = np.exp(y_train_pred_ln)  # 转换回原始价格
y_train_true = np.exp(y_train)  # 转换回原始价格

# 样本外预测
y_test_pred_ln = ols_results.predict(X_test_const)
y_test_pred = np.exp(y_test_pred_ln)  # 转换回原始价格
y_test_true = np.exp(y_test)  # 转换回原始价格

# 计算性能指标
def calculate_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

# 样本内性能
train_mae, train_rmse, train_r2 = calculate_metrics(y_train_true, y_train_pred)

# 样本外性能
test_mae, test_rmse, test_r2 = calculate_metrics(y_test_true, y_test_pred)

# 6折交叉验证
print("\n进行6折交叉验证...")
kf = KFold(n_splits=6, shuffle=True, random_state=42)
cv_scores_mae = []
cv_scores_rmse = []
cv_scores_r2 = []

for train_idx, val_idx in kf.split(X_filled):
    # 分割数据
    X_cv_train, X_cv_val = X_filled.iloc[train_idx], X_filled.iloc[val_idx]
    y_cv_train, y_cv_val = y_filled.iloc[train_idx], y_filled.iloc[val_idx]
    
    # 添加常数项
    X_cv_train_const = sm.add_constant(X_cv_train)
    X_cv_val_const = sm.add_constant(X_cv_val)
    
    # 训练模型
    cv_model = sm.OLS(y_cv_train, X_cv_train_const).fit()
    
    # 预测
    y_cv_pred_ln = cv_model.predict(X_cv_val_const)
    y_cv_pred = np.exp(y_cv_pred_ln)
    y_cv_true = np.exp(y_cv_val)
    
    # 计算指标
    cv_mae, cv_rmse, cv_r2 = calculate_metrics(y_cv_true, y_cv_pred)
    cv_scores_mae.append(cv_mae)
    cv_scores_rmse.append(cv_rmse)
    cv_scores_r2.append(cv_r2)

# 计算交叉验证平均性能
cv_mae_mean = np.mean(cv_scores_mae)
cv_rmse_mean = np.mean(cv_scores_rmse)
cv_r2_mean = np.mean(cv_scores_r2)

# 输出模型结果
print("\n" + "=" * 80)
print("OLS模型结果汇总")
print("=" * 80)
print(ols_results.summary())

# 输出性能指标表格
print("\n" + "=" * 80)
print("模型性能指标")
print("=" * 80)

print("\n性能指标表格:")
print("-" * 80)
print(f"{'Metrics':<25} {'In sample':<15} {'Out of sample':<15} {'Cross-validation':<18}")
print("-" * 80)
print(f"{'R²':<25} {train_r2:.4f}{'':<11} {test_r2:.4f}{'':<11} {cv_r2_mean:.4f}")
print(f"{'MAE':<25} {train_mae:.2f}{'':<11} {test_mae:.2f}{'':<11} {cv_mae_mean:.2f}")
print(f"{'RMSE':<25} {train_rmse:.2f}{'':<11} {test_rmse:.2f}{'':<11} {cv_rmse_mean:.2f}")
print("-" * 80)

# 输出详细的统计信息
print(f"\n详细统计信息:")
print(f"样本数量 (训练集): {len(X_train)}")
print(f"样本数量 (测试集): {len(X_test)}")
print(f"变量数量: {X_train.shape[1]}")
print(f"模型F统计量: {ols_results.fvalue:.2f}")
print(f"F统计量p值: {ols_results.f_pvalue:.4f}")
print(f"AIC: {ols_results.aic:.2f}")
print(f"BIC: {ols_results.bic:.2f}")

# 保存模型结果
model_results = {
    'OLS': {
        'model': ols_results,
        'train_r2': train_r2,
        'test_r2': test_r2,
        'cv_r2': cv_r2_mean,
        'train_mae': train_mae,
        'test_mae': test_mae,
        'cv_mae': cv_mae_mean,
        'train_rmse': train_rmse,
        'test_rmse': test_rmse,
        'cv_rmse': cv_rmse_mean
    }
}

print("\n" + "=" * 80)
print("基础线性回归模型完成")
print("=" * 80)




执行特征工程...
创建连续变量的幂次项...
已创建平方项: area^2
已创建平方项: building_age^2
已创建平方项: greening_rate^2
已创建平方项: plot_ratio^2
已创建平方项: household_total^2
已创建平方项: building_total^2
已创建平方项: property_fee_avg^2
已创建平方项: room_count^2
已创建平方项: total_floor^2
已创建平方项: gas_fee_avg^2
总共处理了 10 个幂次项
城市哑变量已存在
城市距离变量已存在
创建城市和面积的交互项...
已创建交互项: city_area_interaction_city_0
已创建交互项: city_area_interaction_city_1
已创建交互项: city_area_interaction_city_2
已创建交互项: city_area_interaction_city_3
已创建交互项: city_area_interaction_city_4
已创建交互项: city_area_interaction_city_5
已创建交互项: city_area_interaction_city_6
已创建交互项: city_area_interaction_city_7
已创建交互项: city_area_interaction_city_8
已创建交互项: city_area_interaction_city_9
已创建交互项: city_area_interaction_city_10
已创建交互项: city_area_interaction_city_11
总共创建了 12 个城市-面积交互项
创建城市和面积平方的交互项...
已创建交互项: city_area_sq_interaction_city_0
已创建交互项: city_area_sq_interaction_city_1
已创建交互项: city_area_sq_interaction_city_2
已创建交互项: city_area_sq_interaction_city_3
已创建交互项: city_area_sq_interaction_city_4
已创建交互项: city_area_s

In [6]:
# 测试集预测部分
def predict_test_set(model_results, test_data_path='rent_price_final_test_dataset.csv'):
    print("\n" + "=" * 80)
    print("测试集预测")
    print("=" * 80)
    
    # 读取测试集数据
    test_df = pd.read_csv(test_data_path)
    print(f"原始测试集形状: {test_df.shape}")
    
    # 获取训练集的特征列表
    ols_model = model_results['OLS']['model']
    train_features = ols_model.model.exog_names
    train_features_order = train_features[1:]  # 排除常数项
    
    print(f"训练模型使用的特征数量: {len(train_features_order)}")
    
    # 1. 特征工程：创建新变量（与训练集保持一致）
    print("\n执行特征工程...")
    
    # 1.1 创建幂次项
    print("创建连续变量的幂次项...")
    continuous_vars_for_power = [
        'area', 'building_age', 'greening_rate', 'plot_ratio', 'building_total', 'household_total',
        'property_fee_avg', 'room_count', 'total_floor', 'gas_fee_avg'
    ]
    
    power_terms = []
    for var in continuous_vars_for_power:
        if var in test_df.columns:
            # 平方项
            square_term = f'{var}^2'
            test_df[square_term] = test_df[var] ** 2
            power_terms.append(square_term)
            print(f"已创建平方项: {square_term}")
    
    print(f"总共处理了 {len(power_terms)} 个幂次项")
    
    # 1.2 创建城市哑变量（如果测试集中有'城市'列）
    if '城市' in test_df.columns:
        # 删除可能已存在的城市哑变量
        city_dummy_cols = [col for col in test_df.columns if col.startswith('city_')]
        if city_dummy_cols:
            test_df.drop(city_dummy_cols, axis=1, inplace=True)
        
        # 创建城市哑变量
        for city_num in range(12):  # 创建0-11的城市哑变量
            col_name = f'city_{city_num}'
            test_df[col_name] = (test_df['城市'] == city_num).astype(int)
        
        print(f"已创建城市哑变量")
    else:
        print("测试集中没有'城市'列，无法创建城市哑变量")
    
    # 1.3 创建城市距离变量（如果测试集中有'城市'和'距市中心距离_km'列）
    if '城市' in test_df.columns and '距市中心距离_km' in test_df.columns:
        # 删除可能已存在的城市距离变量
        city_dist_cols = [col for col in test_df.columns if col.startswith('city_dist_')]
        if city_dist_cols:
            test_df.drop(city_dist_cols, axis=1, inplace=True)
        
        # 创建城市距离变量
        for city_num in range(12):  # 创建0-11的城市距离变量
            col_name = f'city_dist_{city_num}'
            test_df[col_name] = 0
            # 只有当城市编号匹配时才使用实际距离
            test_df.loc[test_df['城市'] == city_num, col_name] = test_df.loc[test_df['城市'] == city_num, '距市中心距离_km']
        
        print(f"已创建城市距离变量")
    else:
        print("测试集中缺少'城市'或'距市中心距离_km'列，无法创建城市距离变量")

    # 1.4 创建城市和面积的交互项（与训练集保持一致）
    print("创建城市和面积的交互项...")
    
    # 确保城市哑变量和面积变量存在
    city_columns = [f'city_{i}' for i in range(12) if f'city_{i}' in test_df.columns]
    area_column = 'area'
    
    if city_columns and area_column in test_df.columns:
        # 删除可能已存在的交互项
        existing_interaction_cols = [col for col in test_df.columns if col.startswith('city_area_interaction_')]
        if existing_interaction_cols:
            test_df.drop(existing_interaction_cols, axis=1, inplace=True)
            print(f"已删除 {len(existing_interaction_cols)} 个已存在的交互项")
        
        # 创建城市和面积的交互项
        interaction_terms_created = 0
        for city_col in city_columns:
            interaction_col = f'city_area_interaction_{city_col}'
            test_df[interaction_col] = test_df[city_col] * test_df[area_column]
            interaction_terms_created += 1
            print(f"已创建交互项: {interaction_col}")
        
        print(f"总共创建了 {interaction_terms_created} 个城市-面积交互项")
    else:
        if not city_columns:
            print("警告：测试集中未找到城市哑变量，无法创建交互项")
        if area_column not in test_df.columns:
            print("警告：测试集中未找到面积变量，无法创建交互项")
    
    # 1.5 创建城市和面积平方的交互项（可选，与训练集保持一致）
    print("创建城市和面积平方的交互项...")
    area_sq_column = 'area^2'
    
    if city_columns and area_sq_column in test_df.columns:
        # 删除可能已存在的交互项
        existing_interaction_cols = [col for col in test_df.columns if col.startswith('city_area_sq_interaction_')]
        if existing_interaction_cols:
            test_df.drop(existing_interaction_cols, axis=1, inplace=True)
        
        # 创建城市和面积平方的交互项
        interaction_terms_created_sq = 0
        for city_col in city_columns:
            interaction_col = f'city_area_sq_interaction_{city_col}'
            test_df[interaction_col] = test_df[city_col] * test_df[area_sq_column]
            interaction_terms_created_sq += 1
            print(f"已创建交互项: {interaction_col}")
        
        print(f"总共创建了 {interaction_terms_created_sq} 个城市-面积平方交互项")
    else:
        if not city_columns:
            print("警告：测试集中未找到城市哑变量，无法创建面积平方交互项")
        if area_sq_column not in test_df.columns:
            print("警告：测试集中未找到面积平方变量，无法创建交互项")

# 2. 处理缺失值（使用训练集的统计量，应用四层填充逻辑）
# ... [现有的缺失值处理代码] ...
    # 2. 处理缺失值（使用训练集的统计量，应用四层填充逻辑）
    print("\n处理缺失值...")
    
    # 2.1 处理可能影响生成新变量的关键变量
    key_variables = ['城市', '距市中心距离_km']
    for var in key_variables:
        if var in test_df.columns and test_df[var].isnull().sum() > 0:
            null_count = test_df[var].isnull().sum()
            if var == '城市':
                # 对于城市变量，使用训练集的众数填充
                mode_val = df[var].mode()[0] if not df[var].mode().empty else 0
                test_df[var] = test_df[var].fillna(mode_val)
                print(f"  - {var}: {null_count} 个缺失值已用训练集众数 {mode_val} 填充")
            else:
                # 对于距离变量，使用训练集的四层填充逻辑
                null_count_before = test_df[var].isnull().sum()
                
                # 第一层：按板块分组用训练集的中位数填充
                if '板块' in test_df.columns and '板块' in df.columns:
                    # 计算训练集中每个板块的中位数
                    plate_medians = df.groupby('板块')[var].median()
                    
                    # 对于测试集的每个板块，使用训练集对应板块的中位数填充
                    for plate in test_df['板块'].unique():
                        if plate in plate_medians.index and not pd.isna(plate_medians[plate]):
                            # 填充该板块的缺失值
                            plate_mask = (test_df['板块'] == plate) & (test_df[var].isnull())
                            test_df.loc[plate_mask, var] = plate_medians[plate]
                
                # 第二层：检查是否仍有缺失值，如果有则按区县分组用训练集的中位数填充
                null_count_after_plate = test_df[var].isnull().sum()
                if null_count_after_plate > 0 and '区县' in test_df.columns and '区县' in df.columns:
                    # 计算训练集中每个区县的中位数
                    district_medians = df.groupby('区县')[var].median()
                    
                    # 对于测试集的每个区县，使用训练集对应区县的中位数填充
                    for district in test_df['区县'].unique():
                        if district in district_medians.index and not pd.isna(district_medians[district]):
                            # 填充该区县的缺失值
                            district_mask = (test_df['区县'] == district) & (test_df[var].isnull())
                            test_df.loc[district_mask, var] = district_medians[district]
                
                # 第三层：检查是否仍有缺失值，如果有则按城市分组用训练集的中位数填充
                null_count_after_district = test_df[var].isnull().sum()
                if null_count_after_district > 0 and '城市' in test_df.columns and '城市' in df.columns:
                    # 计算训练集中每个城市的中位数
                    city_medians = df.groupby('城市')[var].median()
                    
                    # 对于测试集的每个城市，使用训练集对应城市的中位数填充
                    for city in test_df['城市'].unique():
                        if city in city_medians.index and not pd.isna(city_medians[city]):
                            # 填充该城市的缺失值
                            city_mask = (test_df['城市'] == city) & (test_df[var].isnull())
                            test_df.loc[city_mask, var] = city_medians[city]
                
                # 第四层：检查是否仍有缺失值，如果有则用训练集的全局中位数填充
                null_count_after_city = test_df[var].isnull().sum()
                if null_count_after_city > 0 and var in df.columns:
                    global_median = df[var].median()
                    test_df[var] = test_df[var].fillna(global_median)
                
                null_count_after = test_df[var].isnull().sum()
                print(f"  - {var}: {null_count_before} → {null_count_after} 缺失值")
    
    # 2.2 重新创建可能因缺失值处理而需要更新的衍生变量
    if '城市' in test_df.columns:
        # 重新创建城市哑变量
        for city_num in range(12):
            col_name = f'city_{city_num}'
            test_df[col_name] = (test_df['城市'] == city_num).astype(int)
        print("已重新创建城市哑变量")
    
    if '城市' in test_df.columns and '距市中心距离_km' in test_df.columns:
        # 重新创建城市距离变量
        for city_num in range(12):
            col_name = f'city_dist_{city_num}'
            test_df[col_name] = 0
            test_df.loc[test_df['城市'] == city_num, col_name] = test_df.loc[test_df['城市'] == city_num, '距市中心距离_km']
        print("已重新创建城市距离变量")
    
    # 重新创建城市和面积的交互项（使用填充后的数据）
    if city_columns and area_column in test_df.columns:
        # 重新创建城市和面积的交互项
        for city_col in city_columns:
            interaction_col = f'city_area_interaction_{city_col}'
            test_df[interaction_col] = test_df[city_col] * test_df[area_column]
        print("已重新创建城市-面积交互项")
    
    if city_columns and area_sq_column in test_df.columns:
        # 重新创建城市和面积平方的交互项
        for city_col in city_columns:
            interaction_col = f'city_area_sq_interaction_{city_col}'
            test_df[interaction_col] = test_df[city_col] * test_df[area_sq_column]
        print("已重新创建城市-面积平方交互项")
    
    # 2.3 使用训练集的统计量应用四层填充逻辑填充测试集的连续变量
    continuous_variables_to_fill = [
        'room_count', 'hall_count', '梯数', '户数', 
        'building_age', 'household_total', 'building_total', 'gas_fee_avg',
        'greening_rate', 'plot_ratio', 'property_fee_avg', 'heating_fee_avg', 'parking_spots',
        'total_floor', 'area', '距市中心距离_km','lease_min_months','lease_max_months','lease_avg_months'
    ]
    
    for column in continuous_variables_to_fill:
        if column in test_df.columns and test_df[column].isnull().sum() > 0:
            null_count_before = test_df[column].isnull().sum()
            
            # 第一层：按板块分组用训练集的中位数填充
            if '板块' in test_df.columns and '板块' in df.columns and column in df.columns:
                # 计算训练集中每个板块的中位数
                plate_medians = df.groupby('板块')[column].median()
                
                # 对于测试集的每个板块，使用训练集对应板块的中位数填充
                for plate in test_df['板块'].unique():
                    if plate in plate_medians.index and not pd.isna(plate_medians[plate]):
                        # 填充该板块的缺失值
                        plate_mask = (test_df['板块'] == plate) & (test_df[column].isnull())
                        test_df.loc[plate_mask, column] = plate_medians[plate]
            
            # 第二层：检查是否仍有缺失值，如果有则按区县分组用训练集的中位数填充
            null_count_after_plate = test_df[column].isnull().sum()
            if null_count_after_plate > 0 and '区县' in test_df.columns and '区县' in df.columns and column in df.columns:
                # 计算训练集中每个区县的中位数
                district_medians = df.groupby('区县')[column].median()
                
                # 对于测试集的每个区县，使用训练集对应区县的中位数填充
                for district in test_df['区县'].unique():
                    if district in district_medians.index and not pd.isna(district_medians[district]):
                        # 填充该区县的缺失值
                        district_mask = (test_df['区县'] == district) & (test_df[column].isnull())
                        test_df.loc[district_mask, column] = district_medians[district]
            
            # 第三层：检查是否仍有缺失值，如果有则按城市分组用训练集的中位数填充
            null_count_after_district = test_df[column].isnull().sum()
            if null_count_after_district > 0 and '城市' in test_df.columns and '城市' in df.columns and column in df.columns:
                # 计算训练集中每个城市的中位数
                city_medians = df.groupby('城市')[column].median()
                
                # 对于测试集的每个城市，使用训练集对应城市的中位数填充
                for city in test_df['城市'].unique():
                    if city in city_medians.index and not pd.isna(city_medians[city]):
                        # 填充该城市的缺失值
                        city_mask = (test_df['城市'] == city) & (test_df[column].isnull())
                        test_df.loc[city_mask, column] = city_medians[city]
            
            # 第四层：检查是否仍有缺失值，如果有则用训练集的全局中位数填充
            null_count_after_city = test_df[column].isnull().sum()
            if null_count_after_city > 0 and column in df.columns:
                global_median = df[column].median()
                test_df[column] = test_df[column].fillna(global_median)
            
            null_count_after = test_df[column].isnull().sum()
            print(f"{column}: {null_count_before} → {null_count_after} 缺失值")
    
    # 2.4 重新创建幂次项（确保使用填充后的数据）
    print("\n重新创建幂次项（使用填充后的数据）...")
    for var in continuous_vars_for_power:
        if var in test_df.columns:
            square_term = f'{var}^2'
            test_df[square_term] = test_df[var] ** 2
            print(f"已更新平方项: {square_term}")
    
    # 2.5 对于分类变量和哑变量，使用训练集的统计量应用四层填充逻辑
    categorical_vars_to_fill = [
        'heating_self', 'facility_洗衣机', 'facility_电视', 'facility_空调', 
        'facility_衣柜', 'facility_宽带', 'facility_暖气', 'facility_热水器', 
        'facility_冰箱', 'facility_天然气', 'facility_床', 'pay_bi_monthly', 
        'pay_quarterly', 'pay_semi_annual', 'pay_monthly', 'pay_annual',
        'water_civil', 'water_commercial', 'electricity_commercial', 
        'electricity_civil', 'gas_yes', 'low_dummy', 'middle_dummy', 
        'high_dummy', 'basement_dummy', 'south_dummy', 'north_south_dummy',
        'elevator_yes'
    ]
    
    # 添加其他哑变量
    other_dummy_variables = [
        'decoration_精装', 'decoration_简装', 'decoration_毛坯', 'decoration_其他',
        'top_dummy', 'bottom_dummy', 'transaction_commercial', 'transaction_non_commercial',
        'usage_commercial_office', 'usage_commercial_residential', 'usage_high_end_residential',
        'usage_ordinary_residential', 'usage_other_special',
        'house_age_over_2_years', 'house_age_over_5_years', 'house_age_under_2_years',
        'subway', 'structure_material_brick_concrete', 'structure_material_brick_wood', 
        'structure_material_frame', 'structure_material_mixed', 'structure_material_steel', 
        'structure_material_steel_concrete', 'structure_material_unknown',
        'heating_central', 'property_phone_yes'
    ]
    
    categorical_vars_to_fill.extend(other_dummy_variables)
    
    # 添加城市哑变量到哑变量列表
    city_dummies = [f'city_{i}' for i in range(12)]
    categorical_vars_to_fill.extend(city_dummies)
    
    for column in categorical_vars_to_fill:
        if column in test_df.columns and test_df[column].isnull().sum() > 0:
            null_count_before = test_df[column].isnull().sum()
            
            # 第一层：按板块分组用训练集的众数填充
            if '板块' in test_df.columns and '板块' in df.columns and column in df.columns:
                # 计算训练集中每个板块的众数
                plate_modes = df.groupby('板块')[column].apply(lambda x: x.mode()[0] if not x.mode().empty else 0)
                
                # 对于测试集的每个板块，使用训练集对应板块的众数填充
                for plate in test_df['板块'].unique():
                    if plate in plate_modes.index:
                        # 填充该板块的缺失值
                        plate_mask = (test_df['板块'] == plate) & (test_df[column].isnull())
                        test_df.loc[plate_mask, column] = plate_modes[plate]
            
            # 第二层：检查是否仍有缺失值，如果有则按区县分组用训练集的众数填充
            null_count_after_plate = test_df[column].isnull().sum()
            if null_count_after_plate > 0 and '区县' in test_df.columns and '区县' in df.columns and column in df.columns:
                # 计算训练集中每个区县的众数
                district_modes = df.groupby('区县')[column].apply(lambda x: x.mode()[0] if not x.mode().empty else 0)
                
                # 对于测试集的每个区县，使用训练集对应区县的众数填充
                for district in test_df['区县'].unique():
                    if district in district_modes.index:
                        # 填充该区县的缺失值
                        district_mask = (test_df['区县'] == district) & (test_df[column].isnull())
                        test_df.loc[district_mask, column] = district_modes[district]
            
            # 第三层：检查是否仍有缺失值，如果有则按城市分组用训练集的众数填充
            null_count_after_district = test_df[column].isnull().sum()
            if null_count_after_district > 0 and '城市' in test_df.columns and '城市' in df.columns and column in df.columns:
                # 计算训练集中每个城市的众数
                city_modes = df.groupby('城市')[column].apply(lambda x: x.mode()[0] if not x.mode().empty else 0)
                
                # 对于测试集的每个城市，使用训练集对应城市的众数填充
                for city in test_df['城市'].unique():
                    if city in city_modes.index:
                        # 填充该城市的缺失值
                        city_mask = (test_df['城市'] == city) & (test_df[column].isnull())
                        test_df.loc[city_mask, column] = city_modes[city]
            
            # 第四层：检查是否仍有缺失值，如果有则用训练集的全局众数填充
            null_count_after_city = test_df[column].isnull().sum()
            if null_count_after_city > 0 and column in df.columns:
                global_mode = df[column].mode()[0] if not df[column].mode().empty else 0
                test_df[column] = test_df[column].fillna(global_mode)
            
            null_count_after = test_df[column].isnull().sum()
            print(f"{column}: {null_count_before} → {null_count_after} 缺失值")
    
    # 2.6 最终检查是否还有缺失值
    print("\n最终检查是否还有缺失值...")
    remaining_missing = test_df.isnull().sum().sum()
    if remaining_missing > 0:
        print(f"警告: 测试集仍有 {remaining_missing} 个缺失值")
        missing_cols = test_df.isnull().sum()
        missing_cols = missing_cols[missing_cols > 0]
        print("仍有缺失值的列:")
        for col, count in missing_cols.items():
            print(f"- {col}: {count} 个缺失值")
        
        # 对于仍有缺失值的列，使用0填充
        for col in missing_cols.index:
            test_df[col] = test_df[col].fillna(0)
            print(f"已将 {col} 的缺失值填充为0")
    else:
        print("✅ 测试集已无缺失值")
    
    # 3. 确保测试集包含所有训练模型的特征
    print("\n确保特征一致性...")
    
    # 检查缺失的特征并创建它们（设为0）
    missing_features = set(train_features_order) - set(test_df.columns)
    if missing_features:
        print(f"创建缺失的特征并设为0: {len(missing_features)} 个")
        for feature in missing_features:
            test_df[feature] = 0
            print(f"已创建并设为0: {feature}")
    
    # 确保测试集特征顺序与训练模型一致
    test_base_vars_ordered = [feat for feat in train_features_order if feat in test_df.columns]
    
    print(f"测试集实际使用的特征数量: {len(test_base_vars_ordered)}")
    
    # 4. 进行预测
    print("\n使用OLS模型进行预测...")
    
    # 添加常数项
    X_test_final = test_df[test_base_vars_ordered]
    X_test_final_const = sm.add_constant(X_test_final, has_constant='add')
    
    # 检查预测前的数据
    print(f"预测数据形状: {X_test_final_const.shape}")
    print(f"模型参数数量: {len(ols_model.params)}")
    
    try:
        y_test_pred_ln = ols_model.predict(X_test_final_const)
        
        # 检查预测结果是否有空值
        if pd.isna(y_test_pred_ln).any():
            print(f"警告: 预测结果中有 {pd.isna(y_test_pred_ln).sum()} 个空值")
            # 如果有空值，使用中位数填充
            median_pred = np.nanmedian(y_test_pred_ln)
            y_test_pred_ln = np.where(pd.isna(y_test_pred_ln), median_pred, y_test_pred_ln)
            print(f"已将空值填充为预测值中位数: {median_pred:.4f}")
        
        # 创建只包含ID和预测价格的结果DataFrame
        # 假设测试集中有'ID'列，如果没有，使用索引作为ID
        if 'ID' in test_df.columns:
            result_df = pd.DataFrame({
                'ID': test_df['ID'],
                'Price': np.exp(y_test_pred_ln)  # 将lnPrice转换回原始价格
            })
        else:
            result_df = pd.DataFrame({
                'ID': test_df.index,
                'Price': np.exp(y_test_pred_ln)  # 将lnPrice转换回原始价格
            })
        
        # 检查最终结果是否有空值
        if result_df['Price'].isnull().any():
            print(f"警告: 最终结果中有 {result_df['Price'].isnull().sum()} 个空值")
            # 如果有空值，使用中位数填充
            median_price = result_df['Price'].median()
            result_df['Price'] = result_df['Price'].fillna(median_price)
            print(f"已将空值填充为价格中位数: {median_price:.2f}")
        
        # 保存预测结果
        output_test_path = 'rent_price_test_predictions.csv'
        result_df.to_csv(output_test_path, index=False, float_format='%.2f')
        print(f"\n预测结果已保存到: {output_test_path}")
        print(f"结果文件包含 {len(result_df)} 条预测记录")
        
        # 显示结果的前几行
        print("\n预测结果前5行:")
        print(result_df.head())
        
        # 显示预测结果的统计信息
        print("\n预测结果统计信息:")
        print(f"预测值 Price 范围: [{result_df['Price'].min():.2f}, {result_df['Price'].max():.2f}]")
        print(f"预测值 Price 均值: {result_df['Price'].mean():.2f}")
        print(f"预测值 Price 标准差: {result_df['Price'].std():.2f}")
        
        # 计算Kaggle评分（如果需要）
        # 注意：Kaggle评分通常需要提交到平台获取，这里只是示例
        print("\nKaggle评分: 需要提交到平台获取")
        
        return result_df
        
    except Exception as e:
        print(f"预测过程中出现错误: {e}")
        print("尝试诊断问题...")
        print(f"测试集特征维度: {X_test_final_const.shape}")
        print(f"模型参数维度: {len(ols_model.params)}")
        print(f"测试集特征列: {X_test_final_const.columns.tolist()}")
        print(f"模型参数索引: {ols_model.params.index.tolist()}")
        return None

# 如果需要测试集预测，取消注释下面的行
predict_test_set(model_results)


测试集预测
原始测试集形状: (9773, 56)
训练模型使用的特征数量: 109

执行特征工程...
创建连续变量的幂次项...
已创建平方项: area^2
已创建平方项: building_age^2
已创建平方项: greening_rate^2
已创建平方项: plot_ratio^2
已创建平方项: building_total^2
已创建平方项: household_total^2
已创建平方项: property_fee_avg^2
已创建平方项: room_count^2
已创建平方项: total_floor^2
已创建平方项: gas_fee_avg^2
总共处理了 10 个幂次项
已创建城市哑变量
已创建城市距离变量
创建城市和面积的交互项...
已创建交互项: city_area_interaction_city_0
已创建交互项: city_area_interaction_city_1
已创建交互项: city_area_interaction_city_2
已创建交互项: city_area_interaction_city_3
已创建交互项: city_area_interaction_city_4
已创建交互项: city_area_interaction_city_5
已创建交互项: city_area_interaction_city_6
已创建交互项: city_area_interaction_city_7
已创建交互项: city_area_interaction_city_8
已创建交互项: city_area_interaction_city_9
已创建交互项: city_area_interaction_city_10
已创建交互项: city_area_interaction_city_11
总共创建了 12 个城市-面积交互项
创建城市和面积平方的交互项...
已创建交互项: city_area_sq_interaction_city_0
已创建交互项: city_area_sq_interaction_city_1
已创建交互项: city_area_sq_interaction_city_2
已创建交互项: city_area_sq_interaction_city_3
已创建交互项: city_ar

,ID,Price
0,2000000,1.920099e+05
1,2000001,3.996139e+05
2,2000002,4.115952e+05
3,2000003,1.147955e+06
4,2000004,7.702442e+05
...,...,...
9768,2009768,6.763997e+05
9769,2009769,3.488708e+05
9770,2009770,2.752469e+05
9771,2009771,5.313899e+05
